# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

In [2]:
%dotenv ../05_src/.env

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [3]:
import pypdf
from langchain_core.documents import Document

# Below is a minimal helper for demonstration purposes.
def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]


file_path = "./documents/managing_oneself.pdf"
docs = load_pdf_pages(file_path)
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
#print(document_text)

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [11]:
import sys
import os
sys.path.append('../05_src/')
from utils.clients import get_client

In [12]:
os.getenv('API_GATEWAY_KEY')

'B46WYg0duVfW9k601m7IC1toKcb4ODLD'

In [6]:
from pydantic import BaseModel
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

using course gateway


In [7]:
MODEL

'gpt-4o'

In [8]:


class BookReport(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str
    inputTokens: int
    outputTokens: int

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": "You are a lawyer.  Summarize the user content using legalese."},
        {
            "role": "user",
            "content": f"{document_text}",
        },
    ],
    text_format=BookReport,
)

report = response.output_parsed

In [23]:
report.summary

"In 'Managing Oneself,' Peter F. Drucker posits that individuals must assume responsibility for their careers as employers no longer provide structured career paths. He advises cultivating self-awareness to identify personal strengths, values, and preferences in working styles. Drucker suggests using feedback analysis to discover strengths and improve weaknesses, advocating for focusing energy on areas of competence. Key questions such as 'What are my strengths?' and 'What are my values?' are vital to ensuring alignment with both workplace environments and roles. Drucker emphasizes the importance of knowing where one fits best and how to contribute effectively to achieve career satisfaction and success. Lastly, he discusses the necessity of adapting to life’s second half, suggesting that finding new avenues for contribution and challenge is essential for personal fulfillment."

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [17]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import OpenAIModel
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

#import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
#MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    print("using course gateway")
    model = OpenAIModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    print("NOT using course gateway")
    model = OpenAIModel(model=MODEL, temperature=1)

summary_metric = SummarizationMetric(
    threshold=0.7,
    model=model,   
    include_reason=True,
    assessment_questions= [
        "Does the summary include the top 3 arguements being made?",
        "Does the summary have the same grade-level of writing as the article being summarized?",
        "Does the summary reference the historical difference in work now compared to earlier times?",
        "Does the summary provide actions for the reader to take?",
        "Does the summary include material from the second half of the article?",
        "Does the tone of the summary agree with the tone of the original?"
    ]
)

summary_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

clarity_metric = GEval(
    name="Clarity",
    model=model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding."
    ],
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
)

clarity_test_case = LLMTestCase(
    input=f"{document_text}",
    actual_output=report.summary,
)

using course gateway


In [ ]:
#evaluate(test_cases=[summary_test_case], metrics=[summary_metric])
summary_metric.measure(summary_test_case)
summary_results = {
    "metric_name": summary_metric.__name__,
    "score": summary_metric.score,
    "success": summary_metric.is_successful(),
    "reason": summary_metric.reason
} 

clarity_metric.measure(clarity_test_case)



Output()

0.8868826776238186

In [16]:
import json
summary_json = json.dumps(summary_results, indent=4)
print(summary_json)


{
    "metric_name": "Summarization",
    "score": 0,
    "success": false,
    "reason": "The score is 0.00 because the summary includes extra information not present in the original text, such as the claim that weaknesses are not the focus for building performance. This addition deviates from the true content, resulting in a misleading interpretation of the original message."
}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
